In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import zipfile
import os

# Unzip
zip_path = "/content/drive/MyDrive/NER_Amharic_Finetune/Amharic-E-commerce-Data-Extractor.zip"
extract_path = "/content/all_conll_data"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print(" Folder extracted.")

 Folder extracted.


In [3]:
import os

extract_path = "/content/all_conll_data/Amharic-E-commerce-Data-Extractor"

if os.path.exists(extract_path):
    print(f"Files in {extract_path}:")
    for filename in os.listdir(extract_path):
        print(filename)
else:
    print(f"The directory {extract_path} does not exist.")

Files in /content/all_conll_data/Amharic-E-commerce-Data-Extractor:
logs
.gitignore
.venv
models
.git
.env
scripts
config
reports
notebooks
requirements.txt
README.md
.github
data


In [4]:
import os
from sklearn.model_selection import train_test_split
import json

def parse_conll_file(filepath):
    try:
        with open(filepath, 'r', encoding='utf-8') as f:
            sentences = []
            tokens = []
            tags = []
            for line in f:
                line = line.strip()
                if not line:
                    if tokens:
                        sentences.append({"tokens": tokens, "ner_tags": tags})
                        tokens, tags = [], []
                else:
                    parts = line.split()
                    if len(parts) >= 2:
                        tokens.append(parts[0])
                        tags.append(parts[-1])
            if tokens:
                sentences.append({"tokens": tokens, "ner_tags": tags})
        return sentences
    except UnicodeDecodeError:
        print(f"Skipping file due to encoding error: {filepath}")
        return []

# Set your folder path containing only the annotated .txt/.conll files
data_folder = "/content/all_conll_data/Amharic-E-commerce-Data-Extractor/data/processed"

dataset = []
if os.path.exists(data_folder):
    for filename in os.listdir(data_folder):
        if filename.endswith(".txt") or filename.endswith(".conll"):
            file_path = os.path.join(data_folder, filename)
            print(f"Processing file: {filename}")
            dataset.extend(parse_conll_file(file_path))
else:
    print(f"Directory does not exist: {data_folder}")

print(f"✅ Total examples loaded: {len(dataset)}")

if len(dataset) > 0:
    train_data, val_data = train_test_split(dataset, test_size=0.2, random_state=42)

    output_dir = "/content/drive/MyDrive/NER_Amharic_Finetune/"
    os.makedirs(output_dir, exist_ok=True)

    with open(os.path.join(output_dir, "train.json"), "w", encoding="utf-8") as f:
        json.dump(train_data, f, ensure_ascii=False, indent=2)

    with open(os.path.join(output_dir, "valid.json"), "w", encoding="utf-8") as f:
        json.dump(val_data, f, ensure_ascii=False, indent=2)

    print("✅ Saved train.json and valid.json to Google Drive!")
else:
    print("❌ No data loaded. Skipping train/valid split and saving.")


Processing file: labeled3.conll
Processing file: labeled2.conll
Processing file: labeled_data.txt
✅ Total examples loaded: 131
✅ Saved train.json and valid.json to Google Drive!


In [5]:
!pip install -q transformers datasets seqeval

In [6]:
DATA_DIR = "/content/drive/MyDrive/NER_Amharic_Finetune"
TRAIN_PATH = f"{DATA_DIR}/train.json"
VALID_PATH = f"{DATA_DIR}/valid.json"

In [7]:
# 🧪 Load datasets
from datasets import Dataset

import json

def load_data(path):
    with open(path, 'r', encoding='utf-8') as f:
        return json.load(f)

train_data = load_data(TRAIN_PATH)
valid_data = load_data(VALID_PATH)

train_dataset = Dataset.from_list(train_data)
valid_dataset = Dataset.from_list(valid_data)

print(train_dataset[0])


{'tokens': ['39', '40', '41', '43', '1400', '0933682917', 'ማራኪ'], 'ner_tags': ['O', 'O', 'O', 'O', 'O', 'O', 'O']}


In [8]:
labels = sorted({label for example in train_data for label in example["ner_tags"]})
label2id = {label: idx for idx, label in enumerate(labels)}
id2label = {idx: label for label, idx in label2id.items()}

print(label2id)

{'B-LOC': 0, 'B-PRICE': 1, 'B-PRODUCT': 2, 'B-Product': 3, 'I-LOC': 4, 'I-PRICE': 5, 'I-PRODUCT': 6, 'I-Product': 7, 'O': 8}


In [9]:
from transformers import AutoTokenizer
from transformers import AutoTokenizer, AutoModelForTokenClassification
from transformers import pipeline
tokenizer = AutoTokenizer.from_pretrained("masakhane/afroxlmr-large-ner-masakhaner-1.0_2.0")
model = AutoModelForTokenClassification.from_pretrained("masakhane/afroxlmr-large-ner-masakhaner-1.0_2.0")
nlp = pipeline("ner", model=model, tokenizer=tokenizer)
example = "Emir of Kano turban Zhang wey don spend 18 years for Nigeria"
ner_results = nlp(example)
print(ner_results)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
Device set to use cuda:0


[{'entity': 'B-LOC', 'score': np.float32(0.9999993), 'index': 3, 'word': '▁Kano', 'start': 8, 'end': 12}, {'entity': 'B-PER', 'score': np.float32(0.9999938), 'index': 6, 'word': '▁Z', 'start': 20, 'end': 21}, {'entity': 'I-PER', 'score': np.float32(0.98469293), 'index': 7, 'word': 'hang', 'start': 21, 'end': 25}, {'entity': 'B-DATE', 'score': np.float32(0.9999435), 'index': 12, 'word': '▁18', 'start': 40, 'end': 42}, {'entity': 'I-DATE', 'score': np.float32(0.99999714), 'index': 13, 'word': '▁years', 'start': 43, 'end': 48}, {'entity': 'B-LOC', 'score': np.float32(0.99999905), 'index': 15, 'word': '▁Nigeria', 'start': 53, 'end': 60}]


In [10]:
# 🧼 Tokenization + Label alignment
def tokenize_and_align_labels(example):
    tokenized_inputs = tokenizer(example["tokens"], truncation=True, is_split_into_words=True)
    word_ids = tokenized_inputs.word_ids()

    aligned_labels = []
    previous_word_idx = None
    for word_idx in word_ids:
        if word_idx is None:
            aligned_labels.append(-100)
        elif word_idx != previous_word_idx:
            aligned_labels.append(label2id[example["ner_tags"][word_idx]])
        else:
            aligned_labels.append(label2id[example["ner_tags"][word_idx]] if True else -100)
        previous_word_idx = word_idx

    tokenized_inputs["labels"] = aligned_labels
    return tokenized_inputs

tokenized_train = train_dataset.map(tokenize_and_align_labels)
tokenized_valid = valid_dataset.map(tokenize_and_align_labels)


Map:   0%|          | 0/104 [00:00<?, ? examples/s]

Map:   0%|          | 0/27 [00:00<?, ? examples/s]

In [11]:
# ⚙️ Define model
from transformers import AutoModelForTokenClassification

model = AutoModelForTokenClassification.from_pretrained(
    "masakhane/afroxlmr-large-ner-masakhaner-1.0_2.0",
    num_labels=len(label2id),
    id2label=id2label,
    label2id=label2id,
)

In [12]:
# 🧠 Training setup
from transformers import TrainingArguments, Trainer, DataCollatorForTokenClassification, EvalPrediction
import numpy as np
from seqeval.metrics import accuracy_score, precision_score, recall_score, f1_score

data_collator = DataCollatorForTokenClassification(tokenizer)

def compute_metrics(p: EvalPrediction):
    preds = np.argmax(p.predictions, axis=2)
    labels = p.label_ids

    true_preds = [
        [id2label[p] for (p, l) in zip(pred_row, label_row) if l != -100]
        for pred_row, label_row in zip(preds, labels)
    ]
    true_labels = [
        [id2label[l] for (p, l) in zip(pred_row, label_row) if l != -100]
        for pred_row, label_row in zip(preds, labels)
    ]

    return {
        "precision": precision_score(true_labels, true_preds),
        "recall": recall_score(true_labels, true_preds),
        "f1": f1_score(true_labels, true_preds),
        "accuracy": accuracy_score(true_labels, true_preds),
    }

training_args = TrainingArguments(
    output_dir="/content/drive/MyDrive/NER_Amharic_Model",
    eval_strategy="epoch", # Changed from evaluation_strategy
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=2, # Further Reduced batch size
    per_device_eval_batch_size=8,
    num_train_epochs=5,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=10,
    load_best_model_at_end=True,
    gradient_accumulation_steps=4, # Added gradient accumulation
)

In [13]:
# 🚀 Launch training
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_valid,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()


/tmp/ipython-input-13-81268199.py:2: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.
wandb: Currently logged in as: tsegabogale92 (tsegabogale92-jimma-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,5.320600,0.707705,0.018519,0.010989,0.013793,0.809120
2,0.884000,0.554887,0.046154,0.032967,0.038462,0.830105
3,0.680200,0.354792,0.318841,0.241758,0.275000,0.911622
4,0.371900,0.292531,0.378049,0.340659,0.358382,0.924536
5,0.303000,0.271458,0.426829,0.384615,0.404624,0.929782


TrainOutput(global_step=65, training_loss=1.2678976187339195, metrics={'train_runtime': 460.5175, 'train_samples_per_second': 1.129, 'train_steps_per_second': 0.141, 'total_flos': 114440458472460.0, 'train_loss': 1.2678976187339195, 'epoch': 5.0})

In [15]:
# ✅ Save final model
trainer.save_model("/content/drive/MyDrive/NER_Amharic_Model/final_model")
